In [1]:
import torch
from torch import nn
from torchvision import models

In [23]:
mobilenet = models.mobilenet_v2(num_classes=1081)
d = torch.load(
    '/mnt/c/Users/Utente/Desktop/agro2/Agroalimentary-Quality-Control/models/pretrained/mobilenet_v2_weights_best_acc.tar',
    weights_only=True)
mobilenet.load_state_dict(d['model'])

<All keys matched successfully>

In [24]:
mobilenet.classifier = nn.ModuleList([
    nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(mobilenet.last_channel, 1)
    )
    for _ in range(2)
])

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mobilenet = mobilenet.to(device)

In [ ]:
input_tensor = torch.rand((1, 3, 384, 512))
input_tensor = input_tensor.to(device)

x = mobilenet.features(input_tensor)
x = nn.functional.adaptive_avg_pool2d(x, (1, 1))
x = torch.flatten(x, 1)

outputs = [classifier(x) for classifier in mobilenet.classifier]

In [28]:
torch.cat(outputs, dim=1)

tensor([[ 0.1174, -0.2583]], device='cuda:0', grad_fn=<CatBackward0>)